In [1]:
import numpy as np
import os
from halotools.utils.mcrotations import random_perpendicular_directions
from halotools.utils import elementwise_dot, angles_between_list_of_vectors, normalized_vectors
from ellipse_proj_dumb import compute_ellipse2d_dumb
import opencosmo as oc
import time
import h5py
import yaml
from pathlib import Path

In [2]:
def write_error(f_name, err):
    file_path = "error_lc.txt"
    try:
        with open(file_path, 'a') as file:
            file.write(f"\n\n>>>>> Error on file {f_name}")
            file.write(str(err))
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
    except Exception as e:
        print(f"A file write error occurred: {e}")

def zero_mask(vec):
    nulls = np.ones(len(vec)).astype(bool)
    if len(vec.shape) > 1:
        for i in range(vec.shape[1]):
            nulls &= vec[:,i] == 0
    else:
        nulls &= vec == 0
    return nulls

def nan_mask(vec):
    nans = np.zeros(len(vec)).astype(bool)
    if len(vec.shape) > 1:
        for i in range(vec.shape[1]):
            nans |= np.isnan(vec[:,i])
    else:
        nans |= np.isnan(vec)
    return nans

def sum_zero_vec(vec):
    nulls = zero_mask(vec)
    return sum(nulls)

def bad_mask(vec):
    return zero_mask(vec) | nan_mask(vec)

def check_bad_axes(pos, a, b, c, major, inter, minor):
    print("Nan Counts:")
    print(f"pos: {sum(np.isnan(pos))}")
    print(f"a: {sum(np.isnan(a))}")
    print(f"b: {sum(np.isnan(b))}")
    print(f"c: {sum(np.isnan(c))}")
    print(f"major: {sum(np.isnan(major))}")
    print(f"inter: {sum(np.isnan(inter))}")
    print(f"minor: {sum(np.isnan(minor))}")

    print("\nZero Counts:")
    print(f"pos: {sum_zero_vec(pos)}")
    print(f"a: {sum_zero_vec(a)}")
    print(f"b: {sum_zero_vec(b)}")
    print(f"c: {sum_zero_vec(c)}")
    print(f"major: {sum_zero_vec(major)}")
    print(f"inter: {sum_zero_vec(inter)}")
    print(f"minor: {sum_zero_vec(minor)}")

In [4]:
with open("config.yaml", 'r') as file:
    config = yaml.safe_load(file)

main_dir = Path( config["main_dir"] )
sim = config["sim"]
sim_dir = main_dir / sim

In [5]:
# Select one of the catalogs, and open the data
files = list(f for f in sim_dir.glob("*.hdf5") if f.stem.startswith("lc_cores"))
dataset = oc.open(*files)
N = len(dataset)
print(N)

15683861


In [9]:
# [col for col in dataset.columns if "lsst" in col]

In [10]:
cols = ["top_host_infall_fof_halo_eigS1X", "top_host_infall_fof_halo_eigS1Y", "top_host_infall_fof_halo_eigS1Z",
       "top_host_infall_fof_halo_eigS2X", "top_host_infall_fof_halo_eigS2Y", "top_host_infall_fof_halo_eigS2Z",
       "top_host_infall_fof_halo_eigS3X", "top_host_infall_fof_halo_eigS3Y", "top_host_infall_fof_halo_eigS3Z",
       "x","y","z","central",
       "lsst_g", "lsst_r"]

In [12]:
dataset = dataset.take(N, at="start")
data = dataset.select(cols).get_data("numpy")

major_axis = np.array([data["top_host_infall_fof_halo_eigS1X"], data["top_host_infall_fof_halo_eigS1Y"], data["top_host_infall_fof_halo_eigS1Z"]]).T
inter_axis = np.array([data["top_host_infall_fof_halo_eigS2X"], data["top_host_infall_fof_halo_eigS2Y"], data["top_host_infall_fof_halo_eigS2Z"]]).T
minor_axis = np.array([data["top_host_infall_fof_halo_eigS3X"], data["top_host_infall_fof_halo_eigS3Y"], data["top_host_infall_fof_halo_eigS3Z"]]).T
pos = np.array([data["x"],data["y"],data["z"]]).T
central = np.array(data["central"]) == 1
color_g_min_r = np.array(data["lsst_g"]) - np.array(data["lsst_r"])